# Session 4: Multi-Agent Workflows & Agentic Systems (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 4. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks.

**McKinsey Context:** In this session, we model multi-agent systems after McKinsey engagement teams. Just as a real engagement has a Partner (supervisor), Engagement Manager, Associates (workers), and domain Experts collaborating across workstreams, our multi-agent architectures mirror these consulting team dynamics -- from supervisor-worker delegation to cross-functional handoffs and parallel workstreams.

## Learning Objectives

By the end of this session, students will be able to:

1. **Design multi-agent architectures** using supervisor-worker patterns
2. **Implement agent handoffs** with context preservation
3. **Run agents in parallel** and merge their results
4. **Build collaborative agent teams** for complex tasks
5. **Integrate all Day 2 concepts** into an end-to-end system
6. **Evaluate trade-offs** between single-agent and multi-agent designs

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Literal
import json
import os

print("All imports successful!")

All imports successful!


---
## Demos (Follow Along)

Demos are identical to the student notebook.

### Demo 1: Supervisor-Worker Pattern -- Partner Delegates to Associates

In a McKinsey engagement, the **Partner** acts as the supervisor: they assess incoming client requests and delegate to the right **Associate** based on the nature of the work. One Associate may handle financial analysis, another develops strategic recommendations, and a third builds implementation roadmaps. The Partner reviews all outputs before they reach the client.

This mirrors the **supervisor-worker pattern** in multi-agent systems -- a central routing agent delegates to specialized workers and reviews their deliverables.

In [3]:
# Demo 1 - Supervisor-Worker Pattern: Partner Delegates to Associates

class SupervisorState(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def supervisor_node(state):
    """Partner assesses the engagement request and assigns to the right Associate."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner managing an engagement team. "
            "Based on the client request, assign the work to one Associate: "
            "'financial_analyst' (for financial modeling, valuation, or quantitative analysis), "
            "'strategy_consultant' (for market assessment, competitive strategy, or growth recommendations), or "
            "'operations_expert' (for process optimization, implementation planning, or operational improvements). "
            "Respond with just the associate role name."
        )),
        HumanMessage(content=state["task"])
    ])
    assigned = r.content.strip().lower()
    print(f"  Partner assigned to: {assigned}")
    return {"assigned_to": assigned}

def financial_analyst_node(state):
    """Associate specializing in financial analysis and modeling."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in financial analysis. "
            "Provide structured, data-driven analysis with key metrics, financial implications, "
            "and quantitative insights. Use consulting frameworks where appropriate."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Financial Analysis Workstream] {r.content}"}

def strategy_consultant_node(state):
    """Associate specializing in strategy and market assessment."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in strategy. "
            "Provide market insights, competitive positioning analysis, and strategic recommendations. "
            "Structure your response with clear hypotheses and supporting evidence."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Strategy Workstream] {r.content}"}

def operations_expert_node(state):
    """Associate specializing in operations and implementation."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in operations and implementation. "
            "Provide actionable implementation roadmaps, process improvements, and operational recommendations. "
            "Include timelines, resource requirements, and key milestones."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Operations Workstream] {r.content}"}

def review_node(state):
    """Partner reviews the Associate's deliverable before client presentation."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner reviewing an Associate's deliverable. "
            "Assess the quality, add executive-level perspective, and finalize the recommendation. "
            "Ensure the output is client-ready with clear, actionable insights."
        )),
        HumanMessage(content=f"Client Request: {state['task']}\n\nAssociate Deliverable:\n{state['worker_output']}")
    ])
    return {"final_response": r.content}

def route_worker(state):
    a = state["assigned_to"]
    if "financial" in a: return "financial_analyst"
    if "strategy" in a: return "strategy_consultant"
    return "operations_expert"

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("financial_analyst", financial_analyst_node)
graph.add_node("strategy_consultant", strategy_consultant_node)
graph.add_node("operations_expert", operations_expert_node)
graph.add_node("review", review_node)
graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", route_worker, {
    "financial_analyst": "financial_analyst",
    "strategy_consultant": "strategy_consultant",
    "operations_expert": "operations_expert"
})
for n in ["financial_analyst", "strategy_consultant", "operations_expert"]:
    graph.add_edge(n, "review")
graph.add_edge("review", END)
app = graph.compile()

r = app.invoke({
    "task": "Assess the financial viability of acquiring a mid-size SaaS company with $50M ARR",
    "assigned_to": "", "worker_output": "", "final_response": ""
})
print(f"Partner-Reviewed Deliverable: {r['final_response'][:300]}...")

  Partner assigned to: financial_analyst
Partner-Reviewed Deliverable: ### Executive Assessment of Associate's Deliverable

The deliverable provides a solid foundation for assessing the financial viability of acquiring a mid-size SaaS company with $50 million in ARR. The structured approach is commendable, covering essential areas such as market analysis, financial met...


### Demo 2: Agent-to-Agent Handoff -- Strategy to Operations to Implementation

In consulting engagements, work often flows between specialized teams: the **Strategy team** defines the vision and hypotheses, the **Operations team** translates strategy into operational plans, and the **Implementation team** builds the execution roadmap. Each handoff includes structured context -- engagement scope, key findings, and areas requiring attention.

This mirrors the **agent handoff pattern** where one agent transfers control and context to the next agent in a pipeline.

In [4]:
# Demo 2 - Agent Handoff: Strategy -> Operations -> Implementation

class HandoffState(TypedDict):
    user_request: str
    draft: str
    review_notes: str
    final_output: str
    handoff_context: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def strategy_team_agent(state):
    """Strategy team: defines the strategic vision, hypotheses, and recommendations."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Strategy team. Analyze the client situation and develop "
            "strategic recommendations. Structure your output with: key hypotheses, "
            "supporting evidence, and top-level strategic recommendations."
        )),
        HumanMessage(content=state["user_request"])
    ])
    return {
        "draft": r.content,
        "handoff_context": (
            "Strategy phase complete. Key deliverables: strategic hypotheses validated, "
            "market positioning defined, growth levers identified. "
            "Operations team should focus on translating these into operational plans."
        )
    }

def operations_team_agent(state):
    """Operations team: translates strategy into operational plans and processes."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Operations team. Review the strategy deliverable and "
            "develop an operational plan. Focus on process changes, resource allocation, "
            "organizational structure, and key performance indicators."
        )),
        HumanMessage(content=f"Handoff Context: {state['handoff_context']}\n\nStrategy Deliverable:\n{state['draft']}")
    ])
    return {"review_notes": r.content}

def implementation_team_agent(state):
    """Implementation team: builds the execution roadmap with timelines and milestones."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Implementation team. Take the strategy and operational plans "
            "and create a concrete implementation roadmap. Include: phased timeline, "
            "key milestones, resource requirements, risk mitigation, and success metrics."
        )),
        HumanMessage(content=f"Strategy:\n{state['draft']}\n\nOperational Plan:\n{state['review_notes']}")
    ])
    return {"final_output": r.content}

graph = StateGraph(HandoffState)
graph.add_node("strategy", strategy_team_agent)
graph.add_node("operations", operations_team_agent)
graph.add_node("implementation", implementation_team_agent)
graph.set_entry_point("strategy")
graph.add_edge("strategy", "operations")
graph.add_edge("operations", "implementation")
graph.add_edge("implementation", END)
app = graph.compile()

r = app.invoke({
    "user_request": "Our client, a Fortune 500 retailer, needs to develop a digital transformation strategy to compete with e-commerce disruptors",
    "draft": "", "review_notes": "", "final_output": "", "handoff_context": ""
})
print(f"Implementation Roadmap: {r['final_output'][:300]}...")

Implementation Roadmap: ## Implementation Roadmap for Digital Transformation in Retail

### Phased Timeline

**Phase 1: Assessment and Planning (Months 1-3)**
- Conduct a comprehensive assessment of current capabilities and gaps in omni-channel integration, data analytics, technology, and supply chain.
- Develop a detailed...


### Demo 3: Parallel Agent Execution -- Concurrent Workstreams in M&A Due Diligence

In an M&A due diligence engagement, multiple workstreams run simultaneously: **financial analysis**, **market research**, and **competitive assessment** all proceed in parallel to meet tight deal timelines. Each workstream produces independent findings, and the Engagement Manager synthesizes them into a unified due diligence report.

This mirrors the **parallel execution pattern** -- independent agents work concurrently on different aspects, and a merger agent combines the results.

In [5]:
# Demo 3 - Parallel Agents: M&A Due Diligence Workstreams (sequential simulation)

class ParallelState(TypedDict):
    topic: str
    financial_analysis: str
    market_research: str
    competitive_assessment: str
    merged_report: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def financial_analysis_agent(s):
    """Workstream 1: Financial due diligence -- revenue, margins, projections."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey financial analyst conducting M&A due diligence. "
            "Analyze the financial aspects: revenue trends, margin structure, "
            "cash flow quality, and valuation considerations. Provide 3 key findings."
        )),
        HumanMessage(content=s["topic"])
    ])
    print("  [Financial Analysis Workstream] Complete")
    return {"financial_analysis": r.content}

def market_research_agent(s):
    """Workstream 2: Market due diligence -- TAM, growth drivers, trends."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey market research analyst conducting M&A due diligence. "
            "Assess the target market: total addressable market (TAM), growth drivers, "
            "regulatory landscape, and market trends. Provide 3 key findings."
        )),
        HumanMessage(content=s["topic"])
    ])
    print("  [Market Research Workstream] Complete")
    return {"market_research": r.content}

def competitive_assessment_agent(s):
    """Workstream 3: Competitive due diligence -- positioning, moats, threats."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey competitive strategy analyst conducting M&A due diligence. "
            "Evaluate competitive dynamics: market positioning, competitive moats, "
            "threat of new entrants, and differentiation. Provide 3 key findings."
        )),
        HumanMessage(content=s["topic"])
    ])
    print("  [Competitive Assessment Workstream] Complete")
    return {"competitive_assessment": r.content}

def synthesis_agent(s):
    """Engagement Manager synthesizes all workstreams into a unified due diligence report."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Engagement Manager synthesizing M&A due diligence findings. "
            "Combine the three workstream outputs into a cohesive executive summary. "
            "Include: overall recommendation (proceed/caution/pass), key risks, "
            "synergy opportunities, and critical next steps."
        )),
        HumanMessage(content=(
            f"Deal Context: {s['topic']}\n\n"
            f"Financial Analysis:\n{s['financial_analysis']}\n\n"
            f"Market Research:\n{s['market_research']}\n\n"
            f"Competitive Assessment:\n{s['competitive_assessment']}"
        ))
    ])
    return {"merged_report": r.content}

graph = StateGraph(ParallelState)
for name, fn in [("financial", financial_analysis_agent), ("market", market_research_agent),
                  ("competitive", competitive_assessment_agent), ("synthesis", synthesis_agent)]:
    graph.add_node(name, fn)
graph.set_entry_point("financial")
graph.add_edge("financial", "market")
graph.add_edge("market", "competitive")
graph.add_edge("competitive", "synthesis")
graph.add_edge("synthesis", END)
app = graph.compile()

r = app.invoke({
    "topic": "Acquisition of a mid-market healthcare technology company specializing in AI-powered diagnostics, with $80M revenue and 25% YoY growth",
    "financial_analysis": "", "market_research": "", "competitive_assessment": "", "merged_report": ""
})
print(f"Due Diligence Report:\n{r['merged_report'][:400]}...")

  [Financial Analysis Workstream] Complete
  [Market Research Workstream] Complete
  [Competitive Assessment Workstream] Complete
Due Diligence Report:
### Executive Summary: M&A Due Diligence Findings for Acquisition of AI-Powered Diagnostics Company

#### Overall Recommendation: Proceed with Caution

The acquisition of the mid-market healthcare technology company specializing in AI-powered diagnostics presents a compelling opportunity, given its strong revenue growth and favorable market dynamics. However, careful consideration of key risks and...


### Demo 4: Collaborative Deliverable Creation -- Engagement Team Produces a Client Presentation

In a McKinsey engagement, producing a client deliverable is a collaborative effort. The **Engagement Manager** creates the storyline and structure, an **Associate** develops the detailed content and analyses, and a **Senior Partner** reviews and polishes the final presentation for the Steering Committee.

This mirrors the **collaborative writing pattern** where specialized agents each contribute to building a shared output sequentially.

In [6]:
# Demo 4 - Collaborative Deliverable Creation: Engagement Team Client Presentation

class CollabState(TypedDict):
    topic: str
    outline: str
    draft: str
    polished: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

def em_storyline(s):
    """Engagement Manager creates the presentation storyline and structure."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Engagement Manager creating a presentation storyline. "
            "Develop a clear 3-section structure with the pyramid principle: "
            "start with the answer, then supporting arguments, then evidence. "
            "Include section titles and key messages for each section."
        )),
        HumanMessage(content=f"Create a client presentation storyline for: {s['topic']}")
    ])
    print("  [EM] Storyline created")
    return {"outline": r.content}

def associate_content(s):
    """Associate develops detailed content following the storyline."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate developing content for a client presentation. "
            "Follow the storyline structure and fill in detailed analysis, data points, "
            "and recommendations. Use clear, concise consulting language. "
            "Keep the content executive-ready."
        )),
        HumanMessage(content=f"Engagement Topic: {s['topic']}\n\nStoryline:\n{s['outline']}")
    ])
    print(f"  [Associate] Content developed ({len(r.content)} chars)")
    return {"draft": r.content}

def partner_review(s):
    """Senior Partner polishes the presentation for Steering Committee readiness."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Senior Partner reviewing a client presentation. "
            "Polish the content for C-suite readability: sharpen the 'so what', "
            "ensure each section drives toward actionable recommendations, "
            "and add strategic perspective. Make it Steering Committee-ready."
        )),
        HumanMessage(content=s["draft"])
    ])
    print(f"  [Partner] Presentation polished ({len(r.content)} chars)")
    return {"polished": r.content}

graph = StateGraph(CollabState)
graph.add_node("em_storyline", em_storyline)
graph.add_node("associate_content", associate_content)
graph.add_node("partner_review", partner_review)
graph.set_entry_point("em_storyline")
graph.add_edge("em_storyline", "associate_content")
graph.add_edge("associate_content", "partner_review")
graph.add_edge("partner_review", END)
app = graph.compile()

r = app.invoke({
    "topic": "Cross-functional transformation program for a global pharmaceutical company entering biosimilars",
    "outline": "", "draft": "", "polished": ""
})
print(f"Steering Committee Presentation:\n{r['polished'][:400]}...")

  [EM] Storyline created
  [Associate] Content developed (4800 chars)
  [Partner] Presentation polished (4912 chars)
Steering Committee Presentation:
### Presentation Title: Strategic Cross-Functional Transformation for Biosimilars Market Entry

---

#### Section 1: Strategic Imperative for Entering Biosimilars
**Key Message:** The biosimilars market represents a critical growth opportunity that requires immediate cross-functional transformation to secure our competitive advantage.

- **Market Opportunity:** The global biosimilars market is pro...


### Demo 5: End-to-End Multi-Agent System -- McKinsey Engagement Intake & Delivery

This demo brings together all patterns into a complete engagement delivery system. A **Triage Partner** classifies incoming client requests by type (strategic question, execution task, or relationship touchpoint), routes to the appropriate specialist, and a **Quality Assurance** step ensures deliverable standards before client delivery.

This mirrors how McKinsey manages diverse client interactions through a structured intake and delivery process.

In [7]:
# Demo 5 - End-to-End: McKinsey Engagement Intake & Delivery System

class E2EState(TypedDict):
    user_input: str
    intent: str
    agent_output: str
    quality_score: str
    final_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def classify(s):
    """Triage Partner classifies the client request type."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner triaging client requests. "
            "Classify the request as: 'strategic_question' (needs analysis or recommendation), "
            "'execution_task' (needs an action plan or deliverable), or "
            "'relationship' (general check-in or discussion). "
            "Respond with one classification only."
        )),
        HumanMessage(content=s["user_input"])
    ])
    intent = r.content.strip().lower()
    print(f"  Triage: {intent}")
    return {"intent": intent}

def strategy_advisor(s):
    """Handles strategic questions with hypothesis-driven analysis."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey strategy advisor. Answer the client's strategic question "
            "with structured, hypothesis-driven analysis. Use frameworks like Porter's Five Forces, "
            "value chain analysis, or growth-share matrix as appropriate."
        )),
        HumanMessage(content=s["user_input"])
    ])
    return {"agent_output": r.content}

def execution_lead(s):
    """Handles execution tasks with actionable implementation plans."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey execution lead. Develop a step-by-step action plan "
            "for the client's request. Include timeline, resource requirements, "
            "key milestones, and risk factors."
        )),
        HumanMessage(content=s["user_input"])
    ])
    return {"agent_output": r.content}

def relationship_manager(s):
    """Handles relationship touchpoints with a client-service mindset."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey relationship manager. Respond to the client with warmth "
            "and professionalism. Acknowledge their situation, provide value-adding insights, "
            "and suggest next steps to deepen the engagement."
        )),
        HumanMessage(content=s["user_input"])
    ])
    return {"agent_output": r.content}

def quality_assurance(s):
    """QA check ensures the deliverable meets McKinsey standards."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey quality assurance reviewer. Rate this deliverable 1-10 "
            "on: clarity, actionability, and strategic depth. "
            "Output JSON: {\"score\": N, \"feedback\": \"...\"}"
        )),
        HumanMessage(content=f"Client Request: {s['user_input']}\n\nDeliverable: {s['agent_output']}")
    ])
    return {"quality_score": r.content, "final_output": s["agent_output"]}

def route(s):
    if "strategic" in s["intent"] or "question" in s["intent"]: return "strategy"
    if "execution" in s["intent"] or "task" in s["intent"]: return "execution"
    return "relationship"

graph = StateGraph(E2EState)
for name, fn in [("classify", classify), ("strategy", strategy_advisor),
                  ("execution", execution_lead), ("relationship", relationship_manager),
                  ("quality", quality_assurance)]:
    graph.add_node(name, fn)
graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route, {
    "strategy": "strategy", "execution": "execution", "relationship": "relationship"
})
for n in ["strategy", "execution", "relationship"]:
    graph.add_edge(n, "quality")
graph.add_edge("quality", END)
app = graph.compile()

for inp in [
    "What are the key factors driving consolidation in the European banking sector?",
    "Develop a 90-day plan to reduce supply chain costs by 15%",
    "We enjoyed the last workshop -- can we schedule a follow-up to discuss next steps?"
]:
    print(f"\nClient: {inp}")
    r = app.invoke({"user_input": inp, "intent": "", "agent_output": "", "quality_score": "", "final_output": ""})
    print(f"Deliverable: {r['final_output'][:200]}...")
    print(f"QA Score: {r['quality_score'][:100]}")


Client: What are the key factors driving consolidation in the European banking sector?
  Triage: strategic_question
Deliverable: To analyze the key factors driving consolidation in the European banking sector, we can employ a structured, hypothesis-driven approach using frameworks such as Porter's Five Forces and the value chai...
QA Score: {
  "score": 8,
  "feedback": "The deliverable provides a clear and structured analysis of the facto

Client: Develop a 90-day plan to reduce supply chain costs by 15%
  Triage: execution_task
Deliverable: ### 90-Day Action Plan to Reduce Supply Chain Costs by 15%

#### Objective:
To achieve a 15% reduction in supply chain costs within 90 days through strategic analysis, process optimization, and stakeh...
QA Score: {"score": 8, "feedback": "The deliverable is clear and well-structured, outlining a comprehensive 90

Client: We enjoyed the last workshop -- can we schedule a follow-up to discuss next steps?
  Triage: relationship
Deliverable: Dear [C

---
## Tasks -- Full Solutions

### Task 1: Build a Two-Agent Supervisor-Worker System -- Engagement Manager Routes Client Requests

**Scenario:** A McKinsey Engagement Manager receives diverse client requests during a transformation engagement. Some requests require **quantitative analysis** (financial modeling, data analysis, benchmarking) while others require **strategic advisory** (market entry, competitive positioning, growth strategy). The EM must route each request to the right Associate and review the output before delivering to the client.

**Approach:** The supervisor (Engagement Manager) classifies tasks as quantitative or strategic. Conditional edges route to the right Associate. A Partner review node checks the output quality before client delivery.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 10-12 minutes. Draw the supervisor-worker pattern on the board: Supervisor -> (quantitative OR strategic) -> Partner Review -> END. Emphasize that the supervisor is just an LLM node that classifies and routes -- it doesn't do the work. After the exercise, compare the supervisor pattern to Session 3's conditional routing and discuss when to use each.
>
> **Common Student Mistakes:**
> - The supervisor node doing the actual analysis instead of just classifying and routing
> - Routing function returning 'analyst' when the node is registered as 'quantitative_associate' -- name mismatch
> - Not including the partner review step -- the worker output goes directly to END without quality check
> - Students confusing the supervisor's classification prompt with the worker's analysis prompt
>
> **Skippable?** NO -- CRITICAL -- the supervisor-worker pattern is one of the two core multi-agent architectures (along with handoff). Both Day 3 capstones use variations of this pattern. Do NOT skip.

In [8]:
# Task 1 - SOLUTION: Engagement Manager Routes Client Requests

class Task1State(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def t1_supervisor(state):
    """Engagement Manager assesses the client request and assigns to the right Associate."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Engagement Manager. Classify this client request: "
            "'quantitative' (for financial modeling, data analysis, benchmarking) or "
            "'strategic' (for market entry, competitive positioning, growth strategy). "
            "Respond with one word only."
        )),
        HumanMessage(content=state["task"])
    ])
    assigned = r.content.strip().lower()
    print(f"  EM assigned to: {assigned}")
    return {"assigned_to": assigned}

def t1_quantitative(state):
    """Associate specializing in quantitative analysis and financial modeling."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in quantitative analysis. "
            "Provide structured, data-driven analysis with key metrics, financial projections, "
            "and benchmarking insights. Use frameworks like NPV, IRR, or unit economics as appropriate."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": r.content}

def t1_strategic(state):
    """Associate specializing in strategic advisory and market assessment."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in strategy. "
            "Provide hypothesis-driven strategic recommendations with market sizing, "
            "competitive landscape analysis, and clear 'so what' implications. "
            "Use frameworks like Porter's Five Forces or value chain analysis."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": r.content}

def t1_review(state):
    """Partner reviews the Associate's deliverable for client readiness."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner reviewing an Associate's deliverable. "
            "Assess quality, add executive perspective, and ensure the recommendation "
            "is actionable and client-ready. Finalize the output."
        )),
        HumanMessage(content=f"Client Request: {state['task']}\n\nAssociate Output:\n{state['worker_output']}")
    ])
    return {"final_response": r.content}

def t1_route(state):
    return "quantitative" if "quantitative" in state["assigned_to"] else "strategic"

graph = StateGraph(Task1State)
graph.add_node("supervisor", t1_supervisor)
graph.add_node("quantitative", t1_quantitative)
graph.add_node("strategic", t1_strategic)
graph.add_node("review", t1_review)
graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", t1_route, {"quantitative": "quantitative", "strategic": "strategic"})
graph.add_edge("quantitative", "review")
graph.add_edge("strategic", "review")
graph.add_edge("review", END)
app = graph.compile()

for task in [
    "Build a financial model to assess the ROI of consolidating three regional warehouses into one automated distribution center",
    "Develop a market entry strategy for our client expanding into Southeast Asian consumer electronics"
]:
    r = app.invoke({"task": task, "assigned_to": "", "worker_output": "", "final_response": ""})
    print(f"\nClient Request: {task}")
    print(f"Partner-Reviewed Deliverable: {r['final_response'][:250]}...\n")

  EM assigned to: quantitative

Client Request: Build a financial model to assess the ROI of consolidating three regional warehouses into one automated distribution center
Partner-Reviewed Deliverable: ### Executive Summary

The financial analysis of consolidating three regional warehouses into a single automated distribution center reveals a compelling investment opportunity. The projected Net Present Value (NPV) of $40.88 million, an Internal Rat...

  EM assigned to: strategic

Client Request: Develop a market entry strategy for our client expanding into Southeast Asian consumer electronics
Partner-Reviewed Deliverable: ### Market Entry Strategy for Southeast Asian Consumer Electronics

#### Executive Summary
Our client is poised to expand into the Southeast Asian consumer electronics market, which is ripe for growth due to rising disposable incomes, increasing urba...



### Task 2: Implement Agent Handoff with Context Preservation -- Industry Landscape Assessment Pipeline

**Scenario:** A McKinsey team is conducting an **industry landscape assessment** for a private equity client evaluating investments in the renewable energy sector. The work flows through three phases: a **Research Associate** gathers market data, an **Industry Expert** analyzes the findings and identifies key trends, and a **Partner** synthesizes everything into an investment thesis presentation. Each handoff includes structured context about what was covered and what the next team should focus on.

**Approach:** Each agent reads the accumulated handoff context, does its work, and appends its own notes to the context before passing it along. This ensures downstream agents have full situational awareness.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 10-12 minutes. The key concept is the `handoff_context` field in the state that accumulates notes as work passes between agents. Draw the pipeline: Research Associate -> Industry Expert -> Partner Synthesis, with an arrow showing context flowing forward. After the exercise, discuss: 'What information should be in the handoff? What should NOT be?'
>
> **Common Student Mistakes:**
> - Not accumulating `handoff_context` -- each agent overwrites instead of appending to the previous context
> - Agent B not receiving the output of Agent A -- forgetting to include prior agent output in the prompt
> - Making the handoff_context too verbose (dumping full outputs) instead of summarizing key findings
> - All three agents using the same system prompt -- they should have distinct personas and expertise areas
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. The handoff concept is well-demonstrated in Demo 2. If skipping, show the solution briefly and emphasize the handoff_context accumulation pattern. Task 3 (parallel execution) is more novel and should be prioritized.

In [9]:
# Task 2 - SOLUTION: Industry Landscape Assessment with Context Handoffs

class Task2State(TypedDict):
    topic: str
    research: str
    analysis: str
    presentation: str
    handoff_context: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def t2_research_associate(state):
    """Research Associate gathers market data, industry statistics, and key players."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Research Associate conducting an industry landscape assessment. "
            "Gather key data: market size, growth rates, major players, recent transactions, "
            "and regulatory developments. Structure your findings clearly."
        )),
        HumanMessage(content=state["topic"])
    ])
    ctx = (
        f"[Research Associate] Industry landscape data gathered for: {state['topic']}. "
        "Key areas covered: market sizing, competitive landscape, regulatory environment, "
        "recent M&A activity. Industry Expert should focus on identifying investment themes "
        "and structural trends."
    )
    print("  [Research Associate] Data gathering complete")
    return {"research": r.content, "handoff_context": ctx}

def t2_industry_expert(state):
    """Industry Expert analyzes research and identifies key trends and investment themes."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Industry Expert. Review the research data and identify: "
            "key structural trends, investment themes, potential disruption risks, "
            "and areas of opportunity. Provide a hypothesis-driven analysis."
        )),
        HumanMessage(content=f"Handoff Context: {state['handoff_context']}\n\nResearch Data:\n{state['research']}")
    ])
    ctx = (
        state["handoff_context"] +
        "\n[Industry Expert] Identified 3 key investment themes and 2 structural risks. "
        "Partner should focus on synthesizing into an actionable investment thesis with "
        "clear go/no-go criteria."
    )
    print("  [Industry Expert] Trend analysis complete")
    return {"analysis": r.content, "handoff_context": ctx}

def t2_partner_synthesis(state):
    """Partner synthesizes findings into a client-ready investment thesis presentation."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner. Synthesize the research and analysis into a "
            "compelling investment thesis for the PE client. Include: executive summary, "
            "key investment themes, risk factors, recommended sectors/targets, "
            "and next steps. Make it Steering Committee-ready."
        )),
        HumanMessage(content=f"Handoff Context: {state['handoff_context']}\n\nExpert Analysis:\n{state['analysis']}")
    ])
    print("  [Partner] Investment thesis finalized")
    return {"presentation": r.content}

graph = StateGraph(Task2State)
graph.add_node("research_associate", t2_research_associate)
graph.add_node("industry_expert", t2_industry_expert)
graph.add_node("partner_synthesis", t2_partner_synthesis)
graph.set_entry_point("research_associate")
graph.add_edge("research_associate", "industry_expert")
graph.add_edge("industry_expert", "partner_synthesis")
graph.add_edge("partner_synthesis", END)
app = graph.compile()

r = app.invoke({
    "topic": "Renewable energy sector landscape: solar, wind, and battery storage markets in North America for PE investment screening",
    "research": "", "analysis": "", "presentation": "", "handoff_context": ""
})
print(f"\nInvestment Thesis:\n{r['presentation'][:400]}...")
print(f"\nHandoff Trail:\n{r['handoff_context']}")

  [Research Associate] Data gathering complete
  [Industry Expert] Trend analysis complete
  [Partner] Investment thesis finalized

Investment Thesis:
# Investment Thesis for Renewable Energy Sector in North America

## Executive Summary
The renewable energy sector in North America is experiencing transformative growth, driven by technological advancements, regulatory support, and increasing demand for sustainable energy solutions. This investment thesis outlines a compelling opportunity for private equity (PE) investors to capitalize on the acc...

Handoff Trail:
[Research Associate] Industry landscape data gathered for: Renewable energy sector landscape: solar, wind, and battery storage markets in North America for PE investment screening. Key areas covered: market sizing, competitive landscape, regulatory environment, recent M&A activity. Industry Expert should focus on identifying investment themes and structural trends.
[Industry Expert] Identified 3 key investment themes and 2 st

### Task 3: Create a Parallel Research and Synthesis Pipeline -- Cross-Functional Transformation Assessment

**Scenario:** A McKinsey team is assessing a **cross-functional transformation program** for a global consumer goods company. Three workstreams run in parallel: **Commercial Excellence** (revenue growth, pricing, sales effectiveness), **Supply Chain Optimization** (cost reduction, logistics, procurement), and **Organizational Effectiveness** (talent, culture, operating model). A Senior Partner synthesizes all findings into a unified transformation roadmap.

**Approach:** Three specialist agents each investigate a different dimension. The synthesis agent receives all three outputs and creates a unified transformation report with prioritized recommendations.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 10-12 minutes. Explain that 'parallel' here means fan-out/fan-in: multiple agents process independently, then a synthesis agent combines their outputs. In LangGraph these run sequentially but the key insight is they are *independent* -- no agent needs another's output. After the exercise, discuss: 'How would you make these truly parallel in production?' (Answer: async, threading, or distributed execution).
>
> **Common Student Mistakes:**
> - Making agents dependent on each other's output (sequential) instead of independent (parallel-ready)
> - The synthesis agent not receiving ALL three agent outputs -- only gets the last one
> - Not storing each agent's output in a separate state field (e.g., `commercial_output`, `supply_chain_output`)
> - The synthesis just concatenating outputs instead of truly synthesizing them into a coherent narrative
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. This extends the fan-out/fan-in pattern from Demo 3. If skipping, walk through the solution and discuss how fan-out/fan-in differs from sequential handoff. The concept appears again in Day 3 Track B capstone.

In [10]:
# Task 3 - SOLUTION: Cross-Functional Transformation Assessment

class Task3State(TypedDict):
    topic: str
    commercial_output: str
    supply_chain_output: str
    org_effectiveness_output: str
    report: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def t3_commercial(state):
    """Commercial Excellence workstream: revenue growth, pricing, sales effectiveness."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate leading the Commercial Excellence workstream. "
            "Assess: revenue growth opportunities, pricing optimization potential, "
            "sales force effectiveness, and go-to-market improvements. "
            "Provide 3 key findings with estimated impact."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Commercial Excellence] Workstream complete")
    return {"commercial_output": r.content}

def t3_supply_chain(state):
    """Supply Chain Optimization workstream: cost reduction, logistics, procurement."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate leading the Supply Chain Optimization workstream. "
            "Assess: procurement savings potential, logistics network optimization, "
            "inventory management improvements, and manufacturing efficiency. "
            "Provide 3 key findings with estimated impact."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Supply Chain Optimization] Workstream complete")
    return {"supply_chain_output": r.content}

def t3_org_effectiveness(state):
    """Organizational Effectiveness workstream: talent, culture, operating model."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate leading the Organizational Effectiveness workstream. "
            "Assess: organizational structure alignment, talent gaps and development needs, "
            "cultural enablers and barriers, and operating model improvements. "
            "Provide 3 key findings with estimated impact."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Organizational Effectiveness] Workstream complete")
    return {"org_effectiveness_output": r.content}

def t3_synthesis(state):
    """Senior Partner synthesizes all workstreams into a transformation roadmap."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Senior Partner synthesizing a cross-functional transformation assessment. "
            "Combine the three workstream findings into: an executive summary, "
            "prioritized transformation initiatives (ranked by impact and feasibility), "
            "a phased implementation timeline, and total value at stake. "
            "Make it Steering Committee-ready."
        )),
        HumanMessage(content=(
            f"Transformation Context: {state['topic']}\n\n"
            f"Commercial Excellence:\n{state['commercial_output']}\n\n"
            f"Supply Chain Optimization:\n{state['supply_chain_output']}\n\n"
            f"Organizational Effectiveness:\n{state['org_effectiveness_output']}"
        ))
    ])
    return {"report": r.content}

graph = StateGraph(Task3State)
graph.add_node("commercial", t3_commercial)
graph.add_node("supply_chain", t3_supply_chain)
graph.add_node("org_effectiveness", t3_org_effectiveness)
graph.add_node("synthesis", t3_synthesis)
graph.set_entry_point("commercial")
graph.add_edge("commercial", "supply_chain")
graph.add_edge("supply_chain", "org_effectiveness")
graph.add_edge("org_effectiveness", "synthesis")
graph.add_edge("synthesis", END)
app = graph.compile()

r = app.invoke({
    "topic": "Cross-functional transformation for a $5B global consumer goods company facing margin pressure from private label competitors and rising input costs",
    "commercial_output": "", "supply_chain_output": "", "org_effectiveness_output": "", "report": ""
})
print(f"\nTransformation Roadmap:\n{r['report'][:500]}...")

  [Commercial Excellence] Workstream complete
  [Supply Chain Optimization] Workstream complete
  [Organizational Effectiveness] Workstream complete

Transformation Roadmap:
### Executive Summary

In response to the margin pressures from private label competitors and rising input costs, a comprehensive cross-functional transformation assessment was conducted for our $5B global consumer goods company. The assessment focused on three key workstreams: Commercial Excellence, Supply Chain Optimization, and Organizational Effectiveness. 

The findings indicate significant opportunities for revenue growth and cost savings through targeted initiatives. By leveraging product...


### Task 4: Design and Document a Complete Multi-Agent Architecture -- M&A Due Diligence Review System

**Scenario:** A McKinsey team is supporting a PE client with M&A due diligence on a potential acquisition target. The system uses a **Triage Partner** to assess deal priority, then routes through specialized review agents: **Commercial Due Diligence** (market position, customer concentration, revenue quality), **Operational Due Diligence** (cost structure, scalability, integration complexity), and **Risk Assessment** (regulatory, legal, ESG risks). A final **Deal Committee Report** synthesizes all findings.

**Approach:** We build an M&A due diligence pipeline with conditional routing (the Triage Partner can skip risk assessment for low-priority deals) plus fan-out synthesis. This demonstrates conditional routing + sequential specialists + synthesis.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 15-20 minutes -- this is the Day 2 capstone and most complex exercise. Let students choose their own use case or follow the M&A due diligence example. The key requirements are: 3+ agents, 1+ conditional edge, and a combined output. Walk the room actively. Many students will need help designing the graph before coding. After the exercise, have 2-3 students present their architectures to the class.
>
> **Common Student Mistakes:**
> - Starting to code immediately without designing the graph structure on paper first
> - Creating too many agents (5+) -- the system becomes hard to debug and slow to run
> - Conditional edges that are never triggered because the routing logic doesn't match actual inputs
> - Not having a clear 'output' node that combines all agent results into a final deliverable
> - The state TypedDict growing too large -- each field should serve a clear purpose
>
> **Skippable?** YES -- CAN SKIP if running significantly behind schedule, but try to preserve at least 10 minutes for it. This is the Day 2 capstone. If skipping entirely, walk through the M&A due diligence solution and diagram the architecture. This exercise is important preparation for Day 3 capstones.

In [11]:
# Task 4 - SOLUTION: Complete M&A Due Diligence Review System

print("""Architecture: M&A Due Diligence Multi-Agent System
Agents:
  1. Triage Partner - assesses deal priority and scope of diligence required
  2. Commercial DD Associate - evaluates market position, customers, revenue quality
  3. Operational DD Associate - assesses cost structure, scalability, integration
  4. Risk Assessment Expert - identifies regulatory, legal, and ESG risks
  5. Deal Committee Reporter - synthesizes all findings into investment memo

Flow: triage -> commercial_dd -> operational_dd -> [risk_assessment if critical] -> deal_report -> END
      (triage can skip risk assessment for standard-priority deals)
""")

class DealReviewState(TypedDict):
    deal_description: str
    priority: str
    commercial_dd: str
    operational_dd: str
    risk_assessment: str
    deal_report: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def triage_partner(state):
    """Triage Partner assesses deal priority to determine diligence scope."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner triaging M&A deals. Classify this deal as "
            "'critical' (requires full diligence including risk assessment) or "
            "'standard' (commercial and operational DD sufficient). "
            "Consider deal size, industry risk, and regulatory complexity. One word."
        )),
        HumanMessage(content=state["deal_description"])
    ])
    priority = r.content.strip().lower()
    print(f"  [Triage Partner] Deal priority: {priority}")
    return {"priority": priority}

def commercial_dd_agent(state):
    """Commercial due diligence: market position, revenue quality, customer analysis."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate conducting commercial due diligence. "
            "Evaluate: market positioning and share, customer concentration risk, "
            "revenue quality and sustainability, competitive moats, "
            "and growth trajectory. Be specific and data-oriented."
        )),
        HumanMessage(content=state["deal_description"])
    ])
    return {"commercial_dd": r.content}

def operational_dd_agent(state):
    """Operational due diligence: cost structure, scalability, integration complexity."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate conducting operational due diligence. "
            "Evaluate: cost structure and margin drivers, operational scalability, "
            "technology infrastructure, integration complexity with acquirer, "
            "and key operational risks."
        )),
        HumanMessage(content=state["deal_description"])
    ])
    return {"operational_dd": r.content}

def risk_assessment_agent(state):
    """Risk assessment: regulatory, legal, ESG, and reputational risks."""
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey risk assessment expert conducting enhanced due diligence. "
            "Identify: regulatory and compliance risks, pending or potential litigation, "
            "ESG concerns, reputational risks, and geopolitical exposure. "
            "Rate each risk as high/medium/low."
        )),
        HumanMessage(content=state["deal_description"])
    ])
    return {"risk_assessment": r.content}

def deal_committee_report(state):
    """Synthesize all DD findings into a Deal Committee investment memo."""
    dd_summary = (
        f"Commercial DD:\n{state['commercial_dd']}\n\n"
        f"Operational DD:\n{state['operational_dd']}\n\n"
        f"Risk Assessment:\n{state['risk_assessment'] or 'Not conducted (standard priority deal)'}"
    )
    r = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner preparing the Deal Committee investment memo. "
            "Synthesize all due diligence findings into: executive summary, "
            "key strengths and concerns, overall deal recommendation "
            "(strong proceed / proceed with conditions / pass), "
            "proposed valuation range considerations, and critical next steps."
        )),
        HumanMessage(content=f"Deal:\n{state['deal_description']}\n\nDue Diligence Findings:\n{dd_summary}")
    ])
    return {"deal_report": r.content}

def route_after_operational(state):
    return "risk_assessment" if "critical" in state["priority"] else "deal_report"

graph = StateGraph(DealReviewState)
graph.add_node("triage", triage_partner)
graph.add_node("commercial", commercial_dd_agent)
graph.add_node("operational", operational_dd_agent)
graph.add_node("risk_assessment", risk_assessment_agent)
graph.add_node("deal_report", deal_committee_report)
graph.set_entry_point("triage")
graph.add_edge("triage", "commercial")
graph.add_edge("commercial", "operational")
graph.add_conditional_edges("operational", route_after_operational, {
    "risk_assessment": "risk_assessment", "deal_report": "deal_report"
})
graph.add_edge("risk_assessment", "deal_report")
graph.add_edge("deal_report", END)
app = graph.compile()

# Test with a complex deal requiring full diligence
sample_deal = (
    "Proposed acquisition of MedTech Solutions Inc., a $200M revenue medical device company "
    "specializing in AI-powered surgical robotics. The target operates in 15 countries, "
    "has FDA-pending products, 40% revenue from top 3 hospital networks, "
    "and is facing a patent infringement lawsuit from a competitor. "
    "Asking price: 8x revenue ($1.6B enterprise value)."
)

r = app.invoke({
    "deal_description": sample_deal,
    "priority": "", "commercial_dd": "", "operational_dd": "",
    "risk_assessment": "", "deal_report": ""
})
print(f"\nDeal Committee Investment Memo:\n{r['deal_report']}")

Architecture: M&A Due Diligence Multi-Agent System
Agents:
  1. Triage Partner - assesses deal priority and scope of diligence required
  2. Commercial DD Associate - evaluates market position, customers, revenue quality
  3. Operational DD Associate - assesses cost structure, scalability, integration
  4. Risk Assessment Expert - identifies regulatory, legal, and ESG risks
  5. Deal Committee Reporter - synthesizes all findings into investment memo

Flow: triage -> commercial_dd -> operational_dd -> [risk_assessment if critical] -> deal_report -> END
      (triage can skip risk assessment for standard-priority deals)

  [Triage Partner] Deal priority: critical

Deal Committee Investment Memo:
### Executive Summary
The proposed acquisition of MedTech Solutions Inc., a $200M revenue medical device company specializing in AI-powered surgical robotics, presents a compelling opportunity in a high-growth market. However, the investment carries significant risks, including customer concentra

---
## Summary

In this capstone session students built multi-agent systems modeled after McKinsey engagement teams:

1. **Supervisor-Worker (Partner Delegation)** -- A Partner routes client requests to specialized Associates (financial analyst, strategy consultant, operations expert) and reviews deliverables.
2. **Agent Handoffs (Strategy -> Operations -> Implementation)** -- Structured context passing between consulting phases, mirroring how engagement teams hand off work across workstreams.
3. **Parallel Execution (Concurrent Workstreams)** -- Fan-out/fan-in for independent M&A due diligence workstreams running simultaneously.
4. **Collaborative Deliverables (Engagement Team Presentations)** -- EM storyline, Associate content, and Partner polish -- multiple agents building a client-ready presentation.
5. **End-to-End Systems (Engagement Intake & Delivery)** -- Complete architectures with client request triage, specialist routing, and quality assurance.

**Instructor Notes:**
- Emphasize that multi-agent is not always better -- use the simplest architecture that works. A single well-prompted agent often outperforms a poorly designed multi-agent system.
- For Task 4, encourage students to think about failure modes: what if an Associate produces a weak analysis? How does the Partner catch and redirect?
- Discuss real-world considerations: cost (more agents = more API calls), latency, and debugging complexity. These parallel consulting team costs.
- Draw parallels to McKinsey's actual engagement model: the Partner/EM/Associate hierarchy is a natural supervisor-worker pattern.
- Preview Day 3: production deployment, monitoring, and advanced patterns.

**Day 2 Complete!**